# 02 · CV Pipeline — Detection + Tracking (bbox, no segmentation)

Real video-domain datasets → two YOLO26 **detectors**:
- **flower** (1 class) — separate box per flower.
- **insect** (`bee, fly, beetle, bug, butterfly`) — type is a detection class.

Video: detect flowers (box + stable ID), detect + **track** insects (BoT-SORT,
per-insect colour + ID + majority-voted type), and **count distinct insects landing
per flower per type** (each (insect, flower) once ever), with timestamps.

Idempotent — anything already built/trained is reused. GPU needed for training;
the **last section runs inference from trained weights only (no training)** so
teammates can test directly.

In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
here = Path.cwd()
for cand in [here, *here.parents]:
    if (cand / "src" / "config.py").exists():
        os.chdir(cand); sys.path.insert(0, str(cand)); break
REPO = Path.cwd()
from src import config as C
from src.cv_engine import prepare_detect, train as cvtrain, video_detect
import torch
RUNS = C.INTERIM_DIR / "cv_runs"
def exists(p): return Path(p).exists()
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1 · Build detection datasets (from data/raw)
Reads the downloaded Roboflow COCO + 'flower visits' (Zenodo) sets; rare insect
classes are topped up from iNaturalist. Skipped if already built.

In [ ]:
flower_yaml = C.INTERIM_DIR / "flower_det2" / "data.yaml"
insect_yaml = C.INTERIM_DIR / "insect_multidet" / "data.yaml"
if not flower_yaml.exists(): print(prepare_detect.build_flower())
if not insect_yaml.exists():
    print(prepare_detect.build_insect()); print(prepare_detect.aug_insect_from_inat())
print("flower yaml:", flower_yaml.exists(), "| insect yaml:", insect_yaml.exists())

## 2 · Train the detectors (skip if weights exist)

In [ ]:
flower_w = RUNS / "flower_det2_v2_yolo26m" / "weights" / "best.pt"
insect_w = RUNS / "insect_multidet_v2_yolo26m" / "weights" / "best.pt"
honeybee_w = RUNS / "honeybee_clf" / "best.pt"  # optional honeybee-vs-other-bee flag
if not flower_w.exists():
    cvtrain.train(str(flower_yaml), "flower_det2_v2_yolo26m", model="yolo26m.pt", epochs=100, batch=4, patience=20)
if not insect_w.exists():
    cvtrain.train(str(insect_yaml), "insect_multidet_v2_yolo26m", model="yolo26m.pt", epochs=60, batch=4, patience=20)
print("flower:", exists(flower_w), "| insect:", exists(insect_w))

## 3 · Metrics

In [ ]:
def best_map(run):
    df = pd.read_csv(RUNS / run / "results.csv"); df.columns=[c.strip() for c in df.columns]
    i = df["metrics/mAP50-95(B)"].idxmax()
    return {k: round(float(df.loc[i, f"metrics/{k}(B)"]),4) for k in ["mAP50","mAP50-95","precision","recall"]}
metrics = {"flower": best_map("flower_det2_v2_yolo26m"), "insect": best_map("insect_multidet_v2_yolo26m")}
print(json.dumps(metrics, indent=2))

## 4 · Run on the test videos
Flowers boxed (+ID), insects boxed + tracked (+ID + type), distinct count per flower.

In [ ]:
out = REPO / "test_video_result"; out.mkdir(exist_ok=True)
results = []
for v in sorted(C.TEST_VIDEO_DIR.glob("*.mp4")):
    r = video_detect.count_visits_det(str(v), str(flower_w), str(insect_w), out,
                                        save_video=True,
                                        honeybee_weights=str(honeybee_w) if honeybee_w.exists() else "")
    results.append(r); print(v.name, "->", r["flowers"], "flowers,", r["real_landings"], "real landings @", r["out_fps"], "fps")
# grouped outputs: test_video_result/videos/*.mp4 + test_video_result/csv/*.csv
print("team CSVs:", video_detect.aggregate_csvs(out))   # csv/ALL_landings.csv + csv/ALL_flower_summary.csv

### Visit tallies (per flower, per insect type)

In [ ]:
frames=[]
for f in sorted((out/"csv").glob("*_flower_summary.csv")):
    if f.name.startswith("ALL_"): continue
    frames.append(pd.read_csv(f))
vdf=pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
vdf[vdf["n_real_landings"]>0] if not vdf.empty else vdf   # flowers that got >=1 real landing

### Visit timeline (timestamps)

In [ ]:
tls=[]
for f in sorted((out/"csv").glob("*_landings.csv")):
    if f.name.startswith("ALL_"): continue
    d=pd.read_csv(f)
    if len(d): tls.append(d)
tdf=pd.concat(tls, ignore_index=True) if tls else pd.DataFrame(); tdf.head(20)

### Sample annotated frame

In [ ]:
import cv2, glob
vids=sorted(glob.glob(str(out/"videos"/"*_annotated.mp4")))
if vids:
    cap=cv2.VideoCapture(vids[0]); cap.set(cv2.CAP_PROP_POS_FRAMES,int(cap.get(cv2.CAP_PROP_FRAME_COUNT)*0.5))
    ok,fr=cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(11,6)); plt.axis("off"); plt.title(Path(vids[0]).stem)
        plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); plt.show()

## 5 · ⚡ ONE-SHOT TEST — run **only this cell** (weights ship in the repo)

No training, no dataset build. Loads the committed weights, prints the detector
**metrics**, and runs the full pipeline on **every video in the test-video folder**
(`data/raw/Test_Video/`) → grouped outputs in `test_video_result/videos/` (annotated
mp4s) and `test_video_result/csv/` (per-video + `ALL_*.csv` team tables). This is the
single cell a teammate runs to reproduce everything.

In [ ]:
import sys, os, json
from pathlib import Path
_h = Path.cwd()
for _c in [_h, *_h.parents]:
    if (_c / "src" / "config.py").exists(): os.chdir(_c); sys.path.insert(0, str(_c)); break
import pandas as pd
from src import config as C
from src.cv_engine import video_detect
RUNS = C.INTERIM_DIR / "cv_runs"
flower_w = RUNS / "flower_det2_v2_yolo26m" / "weights" / "best.pt"
insect_w = RUNS / "insect_multidet_v2_yolo26m" / "weights" / "best.pt"
honeybee_w = RUNS / "honeybee_clf" / "best.pt"  # optional honeybee-vs-other-bee flag
assert flower_w.exists() and insect_w.exists(), "weights missing — pull the repo with the committed best.pt files"
def _best(run):
    df = pd.read_csv(RUNS / run / "results.csv"); df.columns = [c.strip() for c in df.columns]
    i = df["metrics/mAP50-95(B)"].idxmax()
    return {k: round(float(df.loc[i, f"metrics/{k}(B)"]), 4) for k in ["mAP50", "mAP50-95", "precision", "recall"]}
metrics = {"flower": _best("flower_det2_v2_yolo26m"), "insect": _best("insect_multidet_v2_yolo26m")}
print("METRICS:", json.dumps(metrics, indent=2))

# run EVERY video in the test-video folder -> test_video_result/{videos,csv}/
out = C.REPO_ROOT / "test_video_result"
vids = sorted(p for p in C.TEST_VIDEO_DIR.iterdir()
              if p.suffix.lower() in (".mp4", ".mov", ".avi", ".mkv"))
results = []
for i, v in enumerate(vids, 1):
    print(f"[{i}/{len(vids)}] {v.name}", flush=True)
    results.append(video_detect.count_visits_det(
        str(v), str(flower_w), str(insect_w), out, save_video=True,
        honeybee_weights=str(honeybee_w) if honeybee_w.exists() else ""))
print("team CSVs:", json.dumps(video_detect.aggregate_csvs(out), indent=2))

## 6 · Summary

In [ ]:
print(json.dumps({"metrics": metrics, "videos": len(results),
                  "real_landings": int(sum(r["real_landings"] for r in results))}, indent=2))